# tune-rect: dual-basis NQS tuning on a 4-point rectangle, judged against QMC

**Campaign 2026-08-05/06, branch `feat/tune-rect`.** Architecture + training search for the
Hadamard-dual `ToricCNN_gridinv` at L=4 OBC on hx ∈ {0.2, 0.6} × hz ∈ {0.1, 0.15}
(inside the topological phase; h_z^c ≈ 0.194, h_x^c = 1), scored against dedicated
ParaToric references on **all observables**, then the winner scaled to L=5, 6 at the
same 4 points (kernel = L−1).

**Winner: dual · noninv (4→8) → inv (8,8) · r=1.05 · dt=0.02→0.002 · ds=1e-3**
(ds=3e-3 required at strong field for L≥5). Everything below reads committed JSONs —
no NetKet required.

In [ ]:
import glob, json, math, os, sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.getcwd().endswith("analysis") else os.getcwd()
sys.path.insert(0, os.path.join(ROOT, "analysis"))
from tuning_table import qmc_reference, COMPARED

POINTS = [(0.2, 0.1), (0.6, 0.1), (0.2, 0.15), (0.6, 0.15)]
PLASMA = {4: plt.cm.plasma(0.15), 5: plt.cm.plasma(0.5), 6: plt.cm.plasma(0.8)}

def openax(ax):
    for s in ("top", "right"): ax.spines[s].set_visible(False)

def load_runs(pattern):
    out = []
    for p in sorted(glob.glob(os.path.join(ROOT, pattern))):
        if p.endswith(".curve.json") or p.endswith(".eval65k.json"): continue
        d = json.load(open(p))
        if "config" not in d or "observables" not in d: continue
        ev = p[:-5] + ".eval65k.json"
        if os.path.exists(ev):
            d["observables"] = {**d["observables"], **json.load(open(ev))["observables"]}
        out.append(d)
    return out
print("helpers ready; ROOT =", ROOT)

## 1 · QMC reference grid (ParaToric, OBC, sum-rule + β-drift validated)

In [ ]:
def qmc_grid():
    rows = []
    for (hx, hz) in POINTS:
        d = os.path.join(ROOT, f"results/qmc_hx{hx}_hz{hz}")
        for L in (4, 5, 6, 7):
            fs = sorted(glob.glob(f"{d}/paratoric_L{L}*_combined.json")) or \
                 sorted(glob.glob(f"{d}/paratoric_L{L}.json")) or \
                 sorted(glob.glob(f"{d}/paratoric_L{L}_beta12_x4_seed*.json"))
            if not fs: continue
            j = json.load(open(fs[0]))
            rows.append((L, hx, hz, j["E"], j["E_err"]))
    print(f"{'L':>2} {'hx':>4} {'hz':>5} {'E_QMC':>12} {'err':>8}")
    for L, hx, hz, E, err in sorted(rows):
        print(f"{L:>2} {hx:>4} {hz:>5} {E:>12.4f} {err:>8.4f}")
    return {(L, hx, hz): (E, e) for L, hx, hz, E, e in rows}
QMC = qmc_grid()

## 2 · L=4 stage-1 standings (30 configs at (0.2, 0.1), 65k-sample re-evaluated)

In [ ]:
tab = json.load(open(os.path.join(ROOT, "results/tune_rect/tuning_table_L4_all.json")))
home = [r for r in tab if tuple(r["point"]) == (0.2, 0.1)]
home.sort(key=lambda r: abs(r["cmp"].get("E0", {}).get("pull", 1e9)))
print(f"{'config':<42} {'relE':>9} {'pullE':>6} {'Vscore':>8} {'params':>7}")
for r in home[:10]:
    c = r["cmp"]["E0"]
    print(f"{r['name'].replace('gridinv_dual_L4_OBC_hx0.2_hz0.1_n2x4_',''):<42} "
          f"{c['rel']:>9.2e} {c['pull']:>+6.1f} {float(r['obs']['Vscore']):>8.1e} {r['n_params']:>7}")

fig, ax = plt.subplots(figsize=(6.5, 4.2))
for r in home:
    c = r["cmp"]["E0"]
    dual = r["dual"]; r09 = float(r["radius"]) < 1.0
    ax.scatter(float(r["obs"]["Vscore"]), abs(c["pull"]),
               s=46, c=[PLASMA[4]], marker="o" if dual else "s",
               edgecolors="k" if r09 else "none", linewidths=1.2, alpha=0.85)
ax.set_xscale("log"); openax(ax)
ax.set_xlabel("Vscore"); ax.set_ylabel("|pull| vs QMC")
ax.set_title("L=4 home point: state quality vs energy accuracy\n(squares = primal; black edge = r0.9)")
plt.tight_layout()
# plt.savefig("figs/tune_rect_L4_quality_vs_accuracy.pdf", bbox_inches="tight", dpi=300)
plt.show()

## 3 · Transfer: 5 architectures × 4 points (L=4, 150 iters, dt=0.02, ds=1e-3)

Cell values: relative energy error vs QMC. The r0.9 (NN-only stencil) class fails at
hx=0.6 — the dropped d=1.0 taps carry the plaquette-adjacent correlations that matter
once ⟨B_p⟩ softens to ≈0.85. One r0.9 run diverged outright; its ds=3e-3 retry trains
cleanly but stays in the failure band → the deficit is capacity, not fragility.

In [ ]:
ARCHS = [("#1 nh4-8+inv8-8 (canonical)", "nh4-8_inv8-8_k3_dt0.02"),
         ("#2 inv8-8-8", "inv8-8-8_k3_dt0.02"),
         ("#3 inv8-8+r0.9", "inv8-8_k3_r0.9_dt0.02"),
         ("#4 nh4-8+inv8-8+r0.9", "nh4-8_inv8-8_k3_r0.9_dt0.02"),
         ("#5 nh4-8+inv8-8-4+r0.9", "nh4-8_inv8-8-4_k3_r0.9_dt0.02")]
M = np.full((len(ARCHS), len(POINTS)), np.nan)
for j, pt in enumerate(POINTS):
    for i, (_, key) in enumerate(ARCHS):
        for r in tab:
            if tuple(r["point"]) == pt and r["name"].endswith(key):
                M[i, j] = r["cmp"]["E0"]["rel"]
fig, ax = plt.subplots(figsize=(7, 3.4))
im = ax.imshow(M, cmap="plasma", aspect="auto")
ax.set_xticks(range(len(POINTS)), [f"({hx},{hz})" for hx, hz in POINTS])
ax.set_yticks(range(len(ARCHS)), [a for a, _ in ARCHS])
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        ax.text(j, i, "DIV" if np.isnan(M[i, j]) else f"{M[i,j]:.1e}",
                ha="center", va="center", fontsize=8,
                color="w" if (np.isnan(M[i,j]) or M[i,j] > np.nanmean(M)) else "k")
ax.set_title("relative energy error vs QMC (L=4)")
plt.colorbar(im, label="rel. err"); plt.tight_layout()
# plt.savefig("figs/tune_rect_transfer_heatmap.pdf", bbox_inches="tight", dpi=300)
plt.show()

## 4 · Scaling the winner: L = 4 → 5 → 6 at all four points (kernel = L−1)

kernel = L−1 grows the invariant convolutions as k³ — 5,345 → 10,377 → 18,673 params at L=4→5→6 — which keeps **params per spin ≈ constant** (37.1 / 34.6 / 34.6). At L≥5 the
hot recipe (ds=1e-3) diverges at hx=0.6 — those points use the ds=3e-3 rescue.

In [ ]:
def scaling_rows():
    rows = []
    for L, pat in [(4, "results/tune_rect/stage1_*/gridinv_dual_L4_*nh4-8_inv8-8_k3_dt0.02.json"),
                   (5, "results/tune_rect/scale_L5_*/gridinv_dual_L5_*.json"),
                   (6, "results/tune_rect/scale_L6_*/gridinv_dual_L6_*.json")]:
        for d in load_runs(pat):
            if d.get("diverged"): continue
            c, o = d["config"], d["observables"]
            pt = (float(c["hx"]), float(c["hz"]))
            ref = QMC.get((L, *pt))
            if not ref: continue
            rel = abs(o["E0"] - ref[0]) / abs(ref[0])
            pull = (o["E0"] - ref[0]) / math.sqrt(o["E_err"]**2 + ref[1]**2)
            rows.append(dict(L=L, pt=pt, rel=rel, pull=pull, V=float(o["Vscore"]),
                             ds=c.get("diag_shift")))
    return rows
S = scaling_rows()
print(f"{'L':>2} {'point':>12} {'relE':>9} {'pull':>6} {'Vscore':>8} {'ds':>7}")
for r in sorted(S, key=lambda r: (r["L"], r["pt"])):
    print(f"{r['L']:>2} {str(r['pt']):>12} {r['rel']:>9.2e} {r['pull']:>+6.1f} {r['V']:>8.1e} {r['ds']:>7}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
MK = {(0.2, 0.1): "o", (0.6, 0.1): "s", (0.2, 0.15): "^", (0.6, 0.15): "D"}
for pt in POINTS:
    d = sorted([r for r in S if r["pt"] == pt], key=lambda r: r["L"])
    if not d: continue
    Ls = [r["L"] for r in d]
    axes[0].plot(Ls, [r["rel"] for r in d], marker=MK[pt], mfc="w", label=f"{pt}",
                 color=plt.cm.plasma(0.15 + 0.25 * POINTS.index(pt)))
    axes[1].plot(Ls, [r["V"] for r in d], marker=MK[pt], mfc="w",
                 color=plt.cm.plasma(0.15 + 0.25 * POINTS.index(pt)))
for ax, yl in zip(axes, ("rel. energy error vs QMC", "Vscore")):
    ax.set_yscale("log"); ax.set_xticks([4, 5, 6]); ax.set_xlabel("L"); ax.set_ylabel(yl); openax(ax)
axes[0].legend(frameon=False, fontsize=8)
plt.suptitle("winner nh(4→8)→inv(8,8), kernel=L−1: capacity strain with L and field")
plt.tight_layout()
# plt.savefig("figs/tune_rect_scaling_vs_L.pdf", bbox_inches="tight", dpi=300)
plt.show()

### 4b · Learning curves — energy per spin, all 4 points × L ∈ {4, 5, 6}

Per-step E/N of the winner (plasma by L) with **two kinds of ribbon, of different
quality**: the **QMC band** (around each dashed reference) is the audited 1σ —
equal-weight scatter over 16–64 independent blocks, sum-rule/β-drift validated;
its width is itself accurate to ~10–20%. The **NQS ribbon** is the per-step
`energy_err` from 1024×8-sample chains, which cannot resolve the autocorrelation
time: calibration against the honest 16-long-chain 65k re-evals shows it
**underestimates the true error by ~2–4×**, and adjacent steps are serially
correlated (persistent sampler). Read the NQS ribbon as an order-of-magnitude
sampling-noise indicator; quantitative comparisons use the re-eval bars in the
tables. Strong-field L=5 curves are the ds=3e-3 rescues; guard-masked steps omitted;
panels zoomed to the plateau region (per-spin refs differ across L via the OBC
boundary-to-bulk ratio).

In [ ]:
def winner_curve(pattern):
    for p in sorted(glob.glob(os.path.join(ROOT, pattern))):
        if p.endswith(".eval65k.json"): continue
        d = json.load(open(p))
        if d.get("diverged"): continue
        cv = d.get("curve") or {}
        E = cv.get("E") or cv.get("energy") or []
        S = cv.get("energy_err") or [0.0] * len(E)
        if E:
            f = lambda a: np.asarray([float(x) if x is not None else np.nan for x in a], float)
            return f(E), f(S)
    return None, None

NSPIN = {4: 144, 5: 300, 6: 540}
fig, axes = plt.subplots(2, 2, figsize=(11, 7.5), sharex=True)
for ax, (hx, hz) in zip(axes.ravel(), POINTS):
    refs = []
    for L, pat in [(4, f"results/tune_rect/stage1_hx{hx}_hz{hz}/gridinv_dual_L4_*nh4-8_inv8-8_k3_dt0.02.json"),
                   (5, f"results/tune_rect/scale_L5_hx{hx}_hz{hz}/gridinv_dual_L5_*.json"),
                   (6, f"results/tune_rect/scale_L6_hx{hx}_hz{hz}/gridinv_dual_L6_*.json")]:
        E, S = winner_curve(pat); ref = QMC.get((L, hx, hz))
        if E is None or ref is None: continue
        N = NSPIN[L]; steps = np.arange(len(E)); m = np.isfinite(E)
        ax.plot(steps[m], E[m] / N, color=PLASMA[L], lw=1.1, label=f"L={L}")
        ax.fill_between(steps[m], (E[m] - S[m]) / N, (E[m] + S[m]) / N,
                        color=PLASMA[L], alpha=0.25, lw=0)          # NQS sampling noise (underest. ~2-4x)
        ax.axhline(ref[0] / N, color=PLASMA[L], ls="--", lw=1.0, alpha=0.85)
        ax.axhspan((ref[0] - ref[1]) / N, (ref[0] + ref[1]) / N,
                   color=PLASMA[L], alpha=0.12, lw=0)               # QMC 1-sigma (honest)
        refs.append(ref[0] / N)
    ax.set_ylim(min(refs) - 0.002, max(refs) + 0.010)   # zoom to the QMC plateaus
    openax(ax)
    ax.set_title(f"(hx, hz) = ({hx}, {hz})", fontsize=10)
for ax in axes[1]: ax.set_xlabel("step")
for ax in axes[:, 0]: ax.set_ylabel(r"$E/N$")
axes[0, 0].legend(frameon=False, fontsize=9)
plt.suptitle("winner learning curves, energy per spin (dashed + band = QMC ref $\\pm1\\sigma$)")
plt.tight_layout()
# plt.savefig("figs/tune_rect_learning_curves.pdf", bbox_inches="tight", dpi=300)
plt.show()

## 5 · Findings & next steps

1. **Dual beats primal at equal budget** once the invariant stack is wide enough and the
   schedule hot: inv(2,2,2)→(8,8) cut the gap 4×; dt 0.01→0.02 converged it within 150
   steps (dt=0.04 spikes early — guard-rescued but strictly worse). Light SR damping
   (ds=1e-3) wins at L=4; **strong field at L=5 requires ds=3e-3** (both hx=0.6 points
   diverged at 1e-3 and retrained cleanly at 3e-3); at L=6 (k=5) ds=1e-3 was stable
   everywhere — the larger invariant kernel appears to smooth the landscape.
2. **L=4 family plateau ≈ +0.04–0.05 above QMC** (rel. 2.3–2.8e-4, QMC-bar-limited) —
   many architectures are equivalent *at L=4*; corners + scaling break the tie.
3. **r0.9 NN-only stencil fails at hx=0.6** (all r0.9 variants +0.21…+0.28 vs
   +0.09…+0.13 for 15-tap; one divergence whose ds=3e-3 retry stays in the failure
   band → capacity, not fragility). Keep 15 taps near the magnetic line.
4. **The winner scales.** nh(4→8)→inv(8,8), kernel=L−1 (params 5,345 / 10,377 / 18,673 — constant ≈35 per spin), rel. err vs QMC:

   | point | L=4 | L=5 | L=6 |
   |---|---|---|---|
   | (0.2,0.10) | 2.3e-4 (+2.5σ) | 2.1e-4 (+2.9σ) | **1.0e-4 (+1.3σ)** |
   | (0.2,0.15) | 2.8e-4 (+2.7σ) | 4.5e-4 (+7.4σ) | **2.9e-5 (+0.7σ)** |
   | (0.6,0.10) | 5.5e-4 (+3.4σ) | 7.6e-4 (+3.8σ)* | 3.6e-4 (+3.0σ) |
   | (0.6,0.15) | 7.0e-4 (+3.2σ) | 8.5e-4 (+6.5σ)* | 6.1e-4 (+4.8σ) |

   (*ds=3e-3.) At L=6 both weak-field points are statistically consistent with QMC;
   accuracy degrades smoothly toward the hard corner. L=6 (kernel 5, 250 iters) *beats* L=5 (kernel 4, 200 iters)
   everywhere — consistent with the kernel=L−1 rule holding per-spin capacity
   constant while the longer schedule converges deeper; the residual gradient
   toward the strong-field corner is the remaining systematic.
5. **Ops findings:** kernel=L−1 rule adopted (user); timing smokes before scaling
   (15.6 s/step L=5, 59.8 s/step L=6 — real nodes ±40% around these); inline O_FM
   estimator returned no value at L=6 (R=3 membranes) — post-hoc extraction via
   tc3d.fm if needed; name-identity rule extended to training knobs after the Phase-B
   collision.
6. **Next:** (a) width-vs-L capacity study at L=6 (inv(12,12)/(16,16) vs the fixed
   winner), (b) seed repeats + 65k re-evals at the corners for honest scaling bars,
   (c) S₂/O_FM transition extraction with the winner along hz at fixed hx, (d) a
   precision-QMC pass (β=24, more blocks) anywhere the NQS is to be certified below
   1e-4 relative.
